## 거래처 계층 구조 테이블 생성
- 오타 정제 → 입고처 제거 → Parent/Child 분류 → Duplicate 통합 → ID 재부여

In [31]:
import msoffcrypto
import io
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# 0. 파일 로드
# ─────────────────────────────────────────────
FILE_PATH = "통합170901DATA취합.xlsx"
PASSWORD  = "33"

with open(FILE_PATH, "rb") as f:
    office_file = msoffcrypto.OfficeFile(f)
    office_file.load_key(password=PASSWORD)
    decrypted = io.BytesIO()
    office_file.decrypt(decrypted)

wb = openpyxl.load_workbook(decrypted, read_only=True, data_only=True)
ws = wb["DATA"]
rows = list(ws.iter_rows(values_only=True))

columns = [
    "key", "영업담당자", "거래처명", "삼일모델명", "업체모델명", "CODE_No",
    "단가", "합의중량_g", "구단가_1",
    "구단가_2", "구단가_3", "구단가_4", "구단가_5", "구단가_6", "구단가_7",
    "대표자", "사업자번호", "주소", "업태", "종목", "납품주소",
    "마감담당자_성명", "마감담당자_이메일", "마감담당자_연락처",
    "실무담당자_성명", "실무담당자_이메일", "실무담당자_연락처",
    "운임", "결제조건_일", "제품군", "비고"
]

data = [row for row in rows[2:] if any(row)]
df_raw = pd.DataFrame(data).iloc[:, :31]
df_raw.columns = columns
df_raw = df_raw.drop(columns=["key"])
df_raw["거래처명"] = df_raw["거래처명"].astype(str).str.strip()
df_raw = df_raw[df_raw["거래처명"].notna() & (df_raw["거래처명"] != "None")].reset_index(drop=True)

print(f"원본 행 수: {len(df_raw)}")
print(f"원본 고유 거래처: {df_raw['거래처명'].nunique()}개")

원본 행 수: 5252
원본 고유 거래처: 900개


In [52]:
# ─────────────────────────────────────────────
# 1. 오타 정제
# 거래처명 전체(슬래시 앞/뒤 포함)에서 오타를 올바른 이름으로 교체
# 추가 오타 발견 시 여기에만 추가하면 됨
# ─────────────────────────────────────────────
TYPO_MAP = {
    "가지채움"      : "가치채움",
    "CJ대한퉁운"    : "CJ대한통운",
    "인펙코리아"    : "인팩코리아",
    "한국파렛트폴"   : "한국파렛트풀",
    "안팩코리아"    :  "인팩코리아",
    "중부스티로폴"  :  "중부스티로폼"
}

def fix_typo(name: str) -> str:
    """슬래시 앞/뒤 각각 오타 수정"""
    if "/" in name:
        parts = name.split("/", 1)
        parts = [TYPO_MAP.get(p.strip(), p.strip()) for p in parts]
        return "/".join(parts)
    return TYPO_MAP.get(name, name)

before = df_raw["거래처명"].copy()
df_raw["거래처명"] = df_raw["거래처명"].apply(fix_typo)

fixed = (before != df_raw["거래처명"]).sum()
print(f"오타 수정된 행: {fixed}개")
print(df_raw.loc[before != df_raw["거래처명"], "거래처명"].value_counts().to_string())

오타 수정된 행: 4개
거래처명
인팩코리아/바이오플레이    4


In [33]:
# ─────────────────────────────────────────────
# 2. 입고처 처리
# 슬래시 앞 토큰이 입고처면 → 앞을 제거하고 뒤(진짜 거래처)만 남김
# ex) 용인공장/한국이노팩  →  한국이노팩
# ─────────────────────────────────────────────
INGGO_SET = {
    "강원공장",
    "둔포공장",
    "용인공장",
    "서해공장",
    "아산공장",
    "싱싱패키지",
}

def strip_inggo(name: str) -> str:
    """입고처가 앞에 있으면 제거하고 뒤 이름만 반환"""
    if "/" in name:
        parent_token = name.split("/")[0].strip()
        if parent_token in INGGO_SET:
            return name.split("/", 1)[1].strip()
    return name

before = df_raw["거래처명"].copy()
df_raw["거래처명"] = df_raw["거래처명"].apply(strip_inggo)

stripped = (before != df_raw["거래처명"]).sum()
print(f"입고처 제거된 행: {stripped}개")
# 변경 전후 확인
changed = before[before != df_raw["거래처명"]]
for old, new in zip(changed.values, df_raw.loc[changed.index, "거래처명"].values):
    print(f"  {old}  →  {new}")

입고처 제거된 행: 213개
  강원공장/한일익스프레스  →  한일익스프레스
  둔포공장/김완수  →  김완수
  둔포공장/김완수  →  김완수
  둔포공장/미래농업연구소  →  미래농업연구소
  둔포공장/미래농업연구소  →  미래농업연구소
  둔포공장/청호수지  →  청호수지
  둔포공장/최동고  →  최동고
  둔포공장/최동고  →  최동고
  둔포공장/최동군  →  최동군
  둔포공장/최동군  →  최동군
  둔포공장/태성산업  →  태성산업
  싱싱패키지/건어맨  →  건어맨
  용인공장/새빛공조  →  새빛공조
  용인공장/우성AFC  →  우성AFC
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  용인공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  강원공장/한일전기  →  한일전기
  용인공장/휴앤로지스  →  휴앤로지스
  용인공장/휴앤로지스  →  휴앤로지스
  용인공장/휴앤로지스  →  휴앤로지스
  용인공장/SHS  →  SHS
  용인공장/SHS  →  SHS
  용인공장/SHS  →  SHS
  용인공장/SHS  →  SHS
  용인공장/KCP이천  →  KCP이천
  용인공장/온세물류  →  온세물류
  용인공장/온세물류  →  온세물류
  용인공장/KCP이천  →  KCP이천
  용인공장/한국이노팩  →  한국이노팩
  용인공장/한국이노팩  →  한국이노팩
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨  →  건어맨
  싱싱패키지/건어맨 

In [34]:
# ─────────────────────────────────────────────
# 3. 메타 정보 추출 (정제 완료된 거래처명 기준)
# ─────────────────────────────────────────────
META_COLS = [
    "거래처명", "대표자", "사업자번호", "주소", "업태", "종목", "납품주소",
    "마감담당자_성명", "마감담당자_이메일", "마감담당자_연락처",
    "실무담당자_성명", "실무담당자_이메일", "실무담당자_연락처",
    "운임", "결제조건_일"
]
df_meta = df_raw[META_COLS].drop_duplicates(subset=["거래처명"]).set_index("거래처명")
print(f"정제 후 고유 거래처: {len(df_meta)}개")

정제 후 고유 거래처: 890개


In [35]:
# ─────────────────────────────────────────────
# 4. Master 테이블 구성
# ─────────────────────────────────────────────
all_clients = set(df_raw["거래처명"].unique())
slash_clients      = {c for c in all_clients if "/" in c}
standalone_clients = {c for c in all_clients if "/" not in c}
parent_names_from_slash = {c.split("/")[0].strip() for c in slash_clients}
auto_create_parents = parent_names_from_slash - standalone_clients

print(f"전체 고유 거래처  : {len(all_clients)}개")
print(f"  슬래시(/) 패턴  : {len(slash_clients)}개")
print(f"  단독 행         : {len(standalone_clients)}개")
print(f"  자동생성 Parent : {len(auto_create_parents)}개")

records = []

# Step A: 단독 행 (고객사)
for name in sorted(standalone_clients):
    meta = df_meta.loc[name].to_dict() if name in df_meta.index else {}
    records.append({
        "거래처명" : name,
        "parent_명": None,
        "node_type": "고객사",
        **{col: meta.get(col) for col in META_COLS[1:]}
    })

# Step B: 자동 생성 Parent (슬래시에만 등장)
for name in sorted(auto_create_parents):
    records.append({
        "거래처명" : name,
        "parent_명": None,
        "node_type": "고객사",
        **{col: None for col in META_COLS[1:]}
    })

# Step C: Child (납품처)
for name in sorted(slash_clients):
    parent_name = name.split("/")[0].strip()
    child_label = name.split("/", 1)[1].strip()
    meta = df_meta.loc[name].to_dict() if name in df_meta.index else {}
    records.append({
        "거래처명" : child_label,
        "parent_명": parent_name,
        "node_type": "납품처",
        **{col: meta.get(col) for col in META_COLS[1:]}
    })

df_master = pd.DataFrame(records).reset_index(drop=True)

# ID 부여
df_master.insert(0, "거래처_ID", range(1, len(df_master) + 1))
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

# parent_id 컬럼 위치 조정
cols = df_master.columns.tolist()
cols.remove("parent_id")
cols.insert(2, "parent_id")
df_master = df_master[cols]

# 정렬 (고객사 먼저, 납품처 뒤)
type_order = {"고객사": 0, "납품처": 1}
df_master["_sort"] = df_master["node_type"].map(type_order)
df_master = df_master.sort_values(["_sort", "거래처_ID"]).drop(columns=["_sort"]).reset_index(drop=True)
df_master["거래처_ID"] = range(1, len(df_master) + 1)
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

전체 고유 거래처  : 890개
  슬래시(/) 패턴  : 252개
  단독 행         : 638개
  자동생성 Parent : 29개


In [ ]:
# ─────────────────────────────────────────────
# 5. Duplicate 통합
# 같은 거래처명이 고객사 + 납품처 둘 다 있는 경우:
#   → 납품처 행은 삭제하지 않고 유지
#   → 거래처_ID를 고객사 ID로 통일
#   → 사업자번호/주소 등 빈 칸은 고객사 데이터로 채우기
# ─────────────────────────────────────────────
고객사_names = set(df_master[df_master["node_type"] == "고객사"]["거래처명"])
납품처_names = set(df_master[df_master["node_type"] == "납품처"]["거래처명"])
duplicates = 고객사_names & 납품처_names

print(f"Duplicate 거래처: {len(duplicates)}개")
for name in sorted(duplicates):
    print(f"  • {name}")

for name in duplicates:
    real_row  = df_master[(df_master["거래처명"] == name) & (df_master["node_type"] == "고객사")].iloc[0]
    real_id   = real_row["거래처_ID"]
    mask_fake = (df_master["거래처명"] == name) & (df_master["node_type"] == "납품처")

    # 납품처 행의 거래처_ID를 고객사 ID로 통일하기 전에 기존 ID 저장
    old_ids = df_master.loc[mask_fake, "거래처_ID"].unique()

    # 거래처_ID를 고객사 ID로 통일
    df_master.loc[mask_fake, "거래처_ID"] = real_id

    # 기존 ID를 parent_id로 가리키던 다른 납품처들도 real_id로 재연결
    for old_id in old_ids:
        mask_children = df_master["parent_id"] == old_id
        df_master.loc[mask_children, "parent_id"] = real_id

    # 빈 칸만 고객사 정보로 채우기
    for col in ["사업자번호", "주소", "대표자", "업태", "종목"]:
        df_master.loc[mask_fake, col] = real_row[col]

print(f"\n정리 후 전체: {len(df_master)}개")
print(f"  고객사: {(df_master['node_type']=='고객사').sum()}개")
print(f"  납품처: {(df_master['node_type']=='납품처').sum()}개")

In [39]:
# ─────────────────────────────────────────────
# 6. 청구대상_ID 추가
# 납품처 → default: parent_id (세금계산서는 parent에게 청구)
# 고객사 → 자기 자신 거래처_ID
# 나중에 예외 거래처는 이 컬럼만 수동 수정하면 됨
# ─────────────────────────────────────────────
df_master["청구대상_ID"] = df_master.apply(
    lambda row: row["parent_id"]
        if (row["node_type"] == "납품처" and pd.notna(row["parent_id"]))
        else row["거래처_ID"],
    axis=1
).astype("Int64")

# parent_id 바로 뒤에 위치
cols = df_master.columns.tolist()
cols.remove("청구대상_ID")
cols.insert(cols.index("parent_id") + 1, "청구대상_ID")
df_master = df_master[cols]

# 샘플 확인
print("【청구대상_ID 샘플 (납품처)】")
sample_cols = ["거래처_ID", "거래처명", "node_type", "parent_명", "parent_id", "청구대상_ID"]
print(df_master[df_master["node_type"] == "납품처"][sample_cols].head(10).to_string(index=False))

【청구대상_ID 샘플 (납품처)】
 거래처_ID     거래처명 node_type parent_명  parent_id  청구대상_ID
    668     보라티알       납품처   AJ네트웍스      639.0      639
    669       정들       납품처   AJ네트웍스      639.0      639
    670       태광       납품처   AJ네트웍스      639.0      639
    671     하겐다즈       납품처   AJ네트웍스      639.0      639
    672     고려포장       납품처  ATEC AP      640.0      640
    673      가미락       납품처   CJ대한통운      641.0      641
    674 강원도옥수수총각       납품처   CJ대한통운      641.0      641
    675    그림엘푸드       납품처   CJ대한통운      641.0      641
    676 남천영농조합법인       납품처   CJ대한통운      641.0      641
    677       도당       납품처   CJ대한통운      641.0      641


In [41]:
# ─────────────────────────────────────────────
# 6. 검증
# ─────────────────────────────────────────────
print("【Parent → Child 샘플】")
for p_name in ["청호나이스", "CJ대한통운", "동원홈푸드", "한국파렛트풀", "한국이노팩", "컬리"]:
    if p_name not in name_to_id:
        print(f"  {p_name} → 없음")
        continue
    pid  = name_to_id[p_name]
    row  = df_master[df_master["거래처_ID"] == pid].iloc[0]
    kids = df_master[df_master["parent_id"] == pid][["거래처_ID","거래처명"]].values
    print(f"  [{pid}] {p_name} ({row['node_type']})")
    for kid_id, kid_name in kids[:5]:
        print(f"       └─ [{int(kid_id)}] {kid_name}")
    if len(kids) > 5:
        print(f"       └─ ... 외 {len(kids)-5}개")

# parent_id가 존재하지 않는 ID를 가리키는 행 확인 (정합성 체크)
valid_ids = set(df_master["거래처_ID"])
broken = df_master[df_master["parent_id"].notna() & ~df_master["parent_id"].isin(valid_ids)]
print(f"\n깨진 parent_id 참조: {len(broken)}개")
if len(broken) > 0:
    print(broken[["거래처_ID","거래처명","parent_id","parent_명"]])

【Parent → Child 샘플】
  [517] 청호나이스 (고객사)
       └─ [853] 미래씨엔에이치
       └─ [854] 에스엠나이스
       └─ [855] 협신산업
  [641] CJ대한통운 (고객사)
       └─ [673] 가미락
       └─ [674] 강원도옥수수총각
       └─ [675] 그림엘푸드
       └─ [676] 남천영농조합법인
       └─ [677] 도당
       └─ ... 외 21개
  [643] 동원홈푸드 (고객사)
       └─ [721] 가산공장
       └─ [722] 금천미트(대전)
       └─ [723] 대전센터
       └─ [724] 시화센터
       └─ [725] 장호원
       └─ ... 외 1개
  [665] 한국파렛트풀 (고객사)
       └─ [891] 대상
       └─ [892] 더다냉동물류
       └─ [893] 더유리빙
       └─ [894] 동원로엑스
       └─ [895] 두레냉동
       └─ ... 외 18개
  [603] 한국이노팩 (고객사)
       └─ [888] 이푸드부산
       └─ [889] 이푸드평택
       └─ [890] 현대그린푸드
  [660] 컬리 (고객사)
       └─ [856] 남양주
       └─ [857] 송파

깨진 parent_id 참조: 0개


In [70]:
df_master[df_master['거래처명'].str.contains('(', regex=False)]

,거래처_ID,거래처명,parent_id,청구대상_ID,parent_명,node_type,대표자,사업자번호,주소,업태,종목,납품주소,마감담당자_성명,마감담당자_이메일,마감담당자_연락처,실무담당자_성명,실무담당자_이메일,실무담당자_연락처,운임,결제조건_일
0,1,(유)돌코리아,NaN,1,NaN,고객사,리차드웨인토만,219-81-28275,"서울특별시 강남구 영동대리511(삼성동,무역센터 트레이드타워12층1204호)","운수창고,도소매","창고업,과실및채소,음,식료품,가공식품,무역(농산물)",경기도 평택시 포승읍 평택항만길 305,윤민아,minayoon@dole.co.kr,010-8298-9424,정일성주임,NaN,010-2209-6748,None,40
1,2,AJ네트웍스(온누리스토어),NaN,2,NaN,고객사,NaN,NaN,충남 아산시 염치읍 서원리 72-16,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,유상호과장,NaN,010-6301-7325,None,None
2,3,AJ네트웍스(한강로직스),NaN,3,NaN,고객사,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,황진구,NaN,NaN,NaN,NaN,010-7682-7411,None,None
110,111,대원제약(정남),NaN,111,NaN,고객사,NaN,124-85-33333,경기도 화성시 정남면 고지리 115-1,제조업,"양약,자양강장제",경기도 화성시 정남면 고지리 115-2,정미선,tax22071@daewonpharm.com,031-353-4451(122),정미선,dwp21200@daewonpharm.com,010-8848-1813,None,30
219,220,본수원갈비(본점),NaN,220,NaN,고객사,NaN,698-86-02930,경기도 수원시 팔달구 중부대로223번길 41,음식및 숙박업,한식점업(갈비),경기도 안성시 신두만곡로 181,NaN,andyyi28@korea.com,031-211-8434,NaN,andyyi28@korea.com,031-211-8434,None,30
278,279,세영FNG (은행나무),NaN,279,NaN,고객사,NaN,469-42-00062,"경기도 수원시 영통구 동수원로 482, 16동 103호(매탄동, 동남상가)",도소매,식자재,경기도 안성시 신두만곡로 181,박진석,progue@naver.com,010-7387-1575,박진석,progue@naver.com,010-7387-1575,"1톤/50,000",0
384,385,연우로지스(선우),NaN,385,NaN,고객사,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,방건준차장,NaN,010-6826-6658,None,None
478,479,주식회사대단(아산),NaN,479,NaN,고객사,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,최재훈,NaN,NaN,NaN,NaN,010-5818-8044,None,None
479,480,주식회사대단(천안),NaN,480,NaN,고객사,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,최재훈,NaN,NaN,NaN,NaN,010-5818-8044,None,None
595,596,하이엠푸드(구서울우유치즈몰안양),NaN,596,NaN,고객사,서창훈,138-02-42779,경기도 의왕시 왕곡동 174-1,도소매,"치즈, 분유",경기도 안성시 신두만곡로 181,NaN,seo6055@naver.com,NaN,NaN,NaN,010-8302-0462,2.5T 7만원,None


In [61]:
# ─────────────────────────────────────────────
# 8. Excel 저장 (Master만 — Detail은 단가 테이블과 함께 추후 작업)
# ─────────────────────────────────────────────
OUTPUT_PATH = "SQL_계층구조_거래처DB.xlsx"

COLORS = {
    "header"      : "1B2A3B",
    "고객사_bg"   : "D6EAF8", "고객사_font": "1A5276",
    "납품처_bg"   : "D5F5E3", "납품처_font": "1E8449",
}

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df_master.to_excel(writer, sheet_name="Master_거래처", index=False)

    ws = writer.sheets["Master_거래처"]

    # 헤더
    h_fill = PatternFill("solid", start_color=COLORS["header"], end_color=COLORS["header"])
    h_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    for cell in ws[1]:
        cell.fill = h_fill
        cell.font = h_font
        cell.alignment = Alignment(horizontal="center", vertical="center")

    # 행 색상 (node_type 기준)
    nt_col = df_master.columns.get_loc("node_type") + 1
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        nt   = ws.cell(row=row_idx, column=nt_col).value
        bg   = COLORS.get(f"{nt}_bg",   "FFFFFF")
        fc   = COLORS.get(f"{nt}_font", "000000")
        bold = (nt == "고객사")
        fill = PatternFill("solid", start_color=bg, end_color=bg)
        font = Font(color=fc, bold=bold, name="Arial", size=9)
        for cell in row:
            cell.fill = fill
            cell.font = font

    # 청구대상_ID 컬럼 강조 (주황색 헤더)
    bill_col = df_master.columns.get_loc("청구대상_ID") + 1
    ws.cell(row=1, column=bill_col).fill = PatternFill("solid", start_color="E67E22", end_color="E67E22")

    # 열 너비
    for col in ws.columns:
        w = max((len(str(c.value)) if c.value else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(w + 4, 45)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

print(f"✅ 저장 완료: {OUTPUT_PATH}")

✅ 저장 완료: SQL_계층구조_거래처DB.xlsx
